In [1]:
!pip install -q google-genai sentence-transformers

In [2]:
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
print("Connected ✅")

Connected ✅


In [4]:
r = client.models.generate_content(model="gemini-3.8-flash", contents="Say hello in 5 words")
print(r.text)

Hello, nice to meet you!


In [5]:
MODELS = ["gemini-3.8-flash", "gemini-3.5-flash", "gemini-3.5-flash-lite"]

In [7]:
import pandas as pd

# Format: (category, [ways customers ask], answer)
FAQ_DATA = [
    # ---------- CARDS ----------
    ("cards", ["How do I report a lost or stolen debit card?", "I lost my card", "My debit card was stolen"],
     "Lock your card instantly in the mobile app under Card Settings, then call 1-800-555-0100 to order a replacement. New cards arrive in 3-5 business days."),
    ("cards", ["How do I activate my new debit card?", "Activate card", "I got a new card in the mail"],
     "Activate your card in the mobile app under Card Settings > Activate Card, or call the number on the sticker on your card. You can also activate it by making a PIN purchase at an ATM."),
    ("cards", ["How do I change my debit card PIN?", "Reset my PIN", "I forgot my card PIN"],
     "You can change your PIN at any Demo Bank ATM or in the mobile app under Card Settings > Change PIN. If you forgot it, you can set a new one in the app after verifying your identity."),
    ("cards", ["How do I lock or unlock my card?", "Freeze my card temporarily", "I found my card, how do I unlock it?"],
     "In the mobile app, go to Card Settings and toggle Lock Card on or off. Locking stops new purchases instantly, and you can unlock it anytime."),
    ("cards", ["Why was my card declined?", "My card isn't working", "Card got rejected at the store"],
     "Common reasons are insufficient funds, a locked card, reaching your daily spending limit, or a fraud alert. Check the app for alerts, or call 1-800-555-0100 for help."),
    ("cards", ["How do I tell the bank I'm traveling?", "Travel notice", "Will my card work abroad?"],
     "Travel notices aren't required, since our fraud system recognizes travel patterns. Make sure your phone number is up to date so we can reach you, and note that foreign transactions have a 3% fee."),
    ("cards", ["What is my daily debit card spending limit?", "Card purchase limit", "How much can I spend per day with my card?"],
     "The standard daily purchase limit is $3,000 and the daily ATM withdrawal limit is $500. You can view your limits in the app under Card Settings."),

    # ---------- FEES ----------
    ("fees", ["How can I avoid the monthly service fee?", "How do I not pay the monthly charge?", "Waive monthly fee"],
     "The $12 monthly fee is waived if you keep a minimum daily balance of $1,500 or receive direct deposits of $500 or more per month."),
    ("fees", ["What is the overdraft fee?", "How much is an overdraft charge?", "I overdrew my account"],
     "The overdraft fee is $34 per item, with a maximum of 3 fees per day. No fee is charged if your account is overdrawn by $50 or less at the end of the day."),
    ("fees", ["How do I avoid overdraft fees?", "Overdraft protection", "Link savings to checking"],
     "Link a savings account for overdraft protection so funds transfer automatically with no fee, and set up low-balance alerts in the app."),
    ("fees", ["Can I get a fee refunded?", "Refund a fee", "Can you reverse the charge on my account?"],
     "Fee refunds are reviewed case by case. Please contact an agent at 1-800-555-0100 or through secure chat in the app."),
    ("fees", ["Is there a fee for foreign transactions?", "International purchase fee", "Fee for using my card abroad"],
     "Yes, purchases in a foreign currency or processed outside the U.S. have a 3% foreign transaction fee."),

    # ---------- ATM ----------
    ("atm", ["Are there fees for using another bank's ATM?", "Out of network ATM fee", "How much does it cost to use a different ATM?"],
     "Out-of-network ATMs charge $3 per withdrawal in the U.S. and $5 internationally, plus any fee the ATM owner charges."),
    ("atm", ["How do I find a nearby ATM?", "ATM near me", "Where is the closest branch?"],
     "Use the ATM & Branch Locator in the mobile app or on our website to find locations, hours, and services near you."),
    ("atm", ["The ATM kept my card", "ATM swallowed my card", "My card is stuck in the ATM"],
     "Lock your card in the app right away for safety, then call 1-800-555-0100. We'll cancel the card and send a replacement in 3-5 business days."),
    ("atm", ["The ATM didn't give me cash but charged my account", "ATM took my money", "Wrong amount from ATM"],
     "Sorry about that. Please report it in the app under Transaction Details > Dispute, or call us. These are usually resolved within 10 business days."),

    # ---------- TRANSFERS & ZELLE ----------
    ("transfers", ["What is the daily Zelle sending limit?", "Zelle limit", "How much can I send with Zelle?"],
     "The standard Zelle limit is $2,000 per day and $10,000 per month. Check your exact limits in the app under Pay & Transfer > Zelle > Limits."),
    ("transfers", ["I sent Zelle money to the wrong person", "Wrong Zelle recipient", "Can I cancel a Zelle payment?"],
     "Zelle payments to enrolled users usually can't be canceled once sent. If the recipient isn't enrolled yet, you can cancel in the app. Otherwise, contact the recipient directly, or call us if you think it was a scam."),
    ("transfers", ["How do I send a wire transfer?", "Domestic wire", "Send money by wire"],
     "Send wires in the app under Pay & Transfer > Wires, or visit a branch. Domestic wires cost $25 and international wires cost $40."),
    ("transfers", ["How long does a transfer between my accounts take?", "Internal transfer time", "Move money between checking and savings"],
     "Transfers between your Demo Bank accounts are instant when made before 11 PM ET."),
    ("transfers", ["How do I transfer money to another bank?", "External transfer", "Send money to my account at another bank"],
     "Link your external account in the app under Pay & Transfer > External Accounts. Standard transfers take 1-3 business days."),

    # ---------- ACCOUNTS ----------
    ("accounts", ["Where do I find my routing number?", "What's the bank routing number?", "Routing number for direct deposit"],
     "Your routing number is shown in the app under Account Details, and at the bottom left of your checks."),
    ("accounts", ["How do I open a new account?", "Open checking account", "I want to open a savings account"],
     "You can open an account online in about 10 minutes or visit any branch. You'll need a government-issued ID and your Social Security number."),
    ("accounts", ["How do I close my account?", "Close my checking account", "I want to cancel my account"],
     "To close an account, call 1-800-555-0100 or visit a branch. Make sure all pending transactions have cleared first."),
    ("accounts", ["What is the savings account interest rate?", "Savings APY", "How much interest does savings earn?"],
     "Savings rates change regularly. Check the current rates on our website or in the app under Explore Products."),
    ("accounts", ["How do I add a joint owner to my account?", "Add someone to my account", "Joint account"],
     "Both people need to visit a branch together with government-issued IDs to add a joint owner."),
    ("accounts", ["How do I update my address or phone number?", "Change my address", "Update contact info"],
     "Update your contact information in the app under Profile & Settings. Changes take effect immediately."),

    # ---------- DEPOSITS ----------
    ("deposits", ["How do I deposit a check with my phone?", "Mobile check deposit", "Deposit a check in the app"],
     "In the app, tap Deposit Checks, sign the back of the check and write 'For mobile deposit only,' then take photos of the front and back."),
    ("deposits", ["When will my deposited check be available?", "Check hold", "Why is my deposit on hold?"],
     "Most check deposits are available the next business day. Larger checks or new accounts may have a hold of up to 5 business days, and you'll be notified if that happens."),
    ("deposits", ["How do I set up direct deposit?", "Direct deposit form", "Get my paycheck deposited"],
     "Download a pre-filled direct deposit form in the app under Account Services > Direct Deposit, and give it to your employer."),
    ("deposits", ["What is the mobile deposit limit?", "How much can I deposit with my phone?", "Mobile deposit maximum"],
     "The standard mobile deposit limit is $5,000 per day and $10,000 per month."),
    ("deposits", ["When does direct deposit arrive?", "My paycheck hasn't shown up", "Direct deposit timing"],
     "Direct deposits usually post by the morning of payday. If yours is missing, confirm with your employer that the payment was sent and that the account details are correct."),

    # ---------- FRAUD & SECURITY ----------
    ("fraud", ["I see a charge I don't recognize. What should I do?", "Someone used my card without my permission", "Unauthorized transaction"],
     "Lock your card in the app right away and call our fraud line at 1-800-555-0199. You can dispute the charge in the app under Transaction Details > Dispute."),
    ("fraud", ["I got a suspicious text or call from the bank", "Is this text from Demo Bank real?", "Phishing email"],
     "Demo Bank will never ask for your password, PIN, or full card number by text, call, or email. Don't click links, and report it to our fraud line at 1-800-555-0199."),
    ("fraud", ["I think I was scammed", "Someone tricked me into sending money", "I paid a scammer"],
     "Please call our fraud line at 1-800-555-0199 immediately. Acting fast gives the best chance of recovering funds."),
    ("fraud", ["How do I set up account alerts?", "Transaction alerts", "Notify me of purchases"],
     "Set up alerts in the app under Profile & Settings > Alerts. You can get notified about purchases, low balances, and large transactions."),
    ("fraud", ["How do I dispute a charge?", "Dispute a transaction", "I was charged twice"],
     "Open the transaction in the app and tap Dispute, or call us. We usually provide a decision within 10 business days."),

    # ---------- ONLINE BANKING ----------
    ("online", ["I forgot my username or password", "Reset password", "Can't log into online banking"],
     "Tap 'Forgot username or password?' on the sign-in screen and follow the steps to verify your identity."),
    ("online", ["My online account is locked", "Too many login attempts", "Account locked out"],
     "For security, accounts lock after several failed sign-in attempts. Use 'Forgot username or password?' to unlock it, or call 1-800-555-0100."),
    ("online", ["How do I enroll in online banking?", "Sign up for online banking", "Create online account"],
     "Download the Demo Bank app or visit our website and tap Sign Up. You'll need your account or card number and Social Security number to verify your identity."),
    ("online", ["How do I get my bank statements?", "Download statement", "Where are my monthly statements?"],
     "View and download up to 7 years of statements in the app under Account Services > Statements & Documents."),
    ("online", ["How do I turn on two-factor authentication?", "Set up 2FA", "Extra login security"],
     "Turn on two-step verification in the app under Profile & Settings > Security."),

    # ---------- CHECKS ----------
    ("checks", ["How do I order checks?", "Order more checks", "I need new checks"],
     "Order checks in the app under Account Services > Order Checks, or at any branch."),
    ("checks", ["How do I stop payment on a check?", "Cancel a check", "Stop a check I wrote"],
     "Request a stop payment in the app under Account Services > Stop Payment. The fee is $30, and the check must not have been cashed yet."),

    # ---------- GENERAL ----------
    ("general", ["What are your branch hours?", "When is the bank open?", "Is the branch open on Saturday?"],
     "Most branches are open 9 AM-5 PM Monday through Friday and 9 AM-1 PM on Saturday. Check the exact hours in the app's Branch Locator."),
    ("general", ["How do I contact customer service?", "Customer service phone number", "Talk to a person"],
     "Call 1-800-555-0100, available 24/7, or use secure chat in the app."),
]

# Expand so each phrasing becomes its own searchable row
rows = []
for faq_id, (category, questions, answer) in enumerate(FAQ_DATA):
    for q in questions:
        rows.append({"faq_id": faq_id, "category": category, "question": q, "answer": answer})

faqs = pd.DataFrame(rows)
print(f"{len(FAQ_DATA)} FAQs, {len(faqs)} searchable phrasings")
faqs.head(10)

46 FAQs, 138 searchable phrasings


,faq_id,category,question,answer
0,0,cards,How do I report a lost or stolen debit card?,Lock your card instantly in the mobile app und...
1,0,cards,I lost my card,Lock your card instantly in the mobile app und...
2,0,cards,My debit card was stolen,Lock your card instantly in the mobile app und...
3,1,cards,How do I activate my new debit card?,Activate your card in the mobile app under Car...
4,1,cards,Activate card,Activate your card in the mobile app under Car...
5,1,cards,I got a new card in the mail,Activate your card in the mobile app under Car...
6,2,cards,How do I change my debit card PIN?,You can change your PIN at any Demo Bank ATM o...
7,2,cards,Reset my PIN,You can change your PIN at any Demo Bank ATM o...
8,2,cards,I forgot my card PIN,You can change your PIN at any Demo Bank ATM o...
9,3,cards,How do I lock or unlock my card?,"In the mobile app, go to Card Settings and tog..."


In [17]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("all-mpnet-base-v2")
faq_embeddings = embedder.encode(faqs["question"].tolist(), convert_to_tensor=True)

def retrieve(question, top_k=5):
    q_emb = embedder.encode(question, convert_to_tensor=True)
    hits = util.semantic_search(q_emb, faq_embeddings, top_k=top_k)[0]
    return [(faqs.iloc[h["corpus_id"]], h["score"]) for h in hits]

# Quick test
for row, score in retrieve("someone used my card without permission"):
    print(f"{score:.2f}  {row['question']}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

1.00  Someone used my card without my permission
0.66  My debit card was stolen
0.57  How do I report a lost or stolen debit card?
0.55  The ATM kept my card
0.54  ATM swallowed my card


In [14]:
import re
import time
from google.genai import errors

SENSITIVE = re.compile(r"\b\d{9,16}\b|ssn|social security|password|\bpin\b", re.I)

SYSTEM_PROMPT = """You are BankBuddy, a customer support assistant for Demo Bank.
Rules:
- Answer ONLY using the FAQ entries provided. Never invent fees, limits, or policies.
- If the FAQs don't answer the question, say: "I'm not sure about that. Let me connect you with an agent."
- Never ask for account numbers, SSNs, PINs, or passwords.
- For fraud or stolen cards, tell the customer to act immediately.
- Be friendly and concise (2-4 sentences)."""

# Tried in order: if one is busy or unavailable, the next one is used
MODELS = ["gemini-3.5-flash-lite", "gemini-3.5-flash", "gemini-3.8-flash"]

def call_gemini(prompt):
    for model in MODELS:
        for attempt in range(2):   # 2 tries per model
            try:
                response = client.models.generate_content(
                    model=model,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        temperature=0.2,
                    ),
                )
                return response.text
            except errors.ServerError:      # 503 busy: wait, then retry
                time.sleep(2 * (attempt + 1))
            except errors.ClientError as e:  # model unavailable or quota: try next model
                print(f"(skipping {model}: {e.code})")
                break
    return "Sorry, our assistant is temporarily unavailable. Please try again shortly or contact an agent."

def ask_bot(question):
    if SENSITIVE.search(question):
        return "⚠️ For your security, please never share account numbers, SSNs, PINs, or passwords in chat."

    results = retrieve(question, top_k=5)
    if results[0][1] < 0.35:   # nothing relevant found
        return "I'm not sure about that. Let me connect you with an agent."

    # Remove duplicate answers (several phrasings share one FAQ)
    seen, unique = set(), []
    for r, _ in results:
        if r["faq_id"] not in seen:
            seen.add(r["faq_id"])
            unique.append(r)

    context = "\n\n".join(f"Q: {r['question']}\nA: {r['answer']}" for r in unique)
    return call_gemini(f"FAQ entries:\n{context}\n\nCustomer question: {question}")

In [15]:
import textwrap

tests = [
    "I lost my debit card at the mall",
    "how do i not pay the monthly charge",
    "my account number is 123456789012, can you check my balance?",
    "what's the mortgage rate today?",      # not in FAQs, should hand off
    "atm ate my card",
    "got a weird text saying it's from the bank",
]

for q in tests:
    print(f"🧑 {q}")
    print("🤖 " + textwrap.fill(ask_bot(q), width=100) + "\n")

🧑 I lost my debit card at the mall
🤖 I'm sorry to hear that! Please lock your card instantly in the mobile app under Card Settings, then
call 1-800-555-0100 to order a replacement. New cards typically arrive in 3-5 business days.

🧑 how do i not pay the monthly charge
🤖 The $12 monthly fee is waived if you keep a minimum daily balance of $1,500 or receive direct
deposits of $500 or more per month. Let me know if you have any other questions!

🧑 my account number is 123456789012, can you check my balance?
🤖 ⚠️ For your security, please never share account numbers, SSNs, PINs, or passwords in chat.

🧑 what's the mortgage rate today?
🤖 I'm not sure about that. Let me connect you with an agent.

🧑 atm ate my card
🤖 Lock your card in the app right away for safety, then call 1-800-555-0100. We'll cancel the card and
send a replacement in 3-5 business days.

🧑 got a weird text saying it's from the bank
🤖 Demo Bank will never ask for your password, PIN, or full card number by text, call, or em

In [18]:
# ---------- 1. RETRIEVAL ACCURACY ----------
# (new question, expected faq_id) - none of these phrasings are in the FAQ data
retrieval_tests = [
    ("my wallet got stolen with my card in it", 0),
    ("how do i turn on the card i just received", 1),
    ("i want a new pin number for my debit card", 2),
    ("can i pause my card for a while", 3),
    ("why does my card keep getting denied at the gas station", 4),
    ("going to mexico next week, do i need to tell you", 5),
    ("what's the max i can take out of an atm per day", 6),
    ("why am i being charged 12 dollars every month", 7),
    ("how much do you charge when i spend more than i have", 8),
    ("can money move from savings automatically if checking is low", 9),
    ("please give me my overdraft fee back", 10),
    ("i used another bank's machine, how much will it cost", 12),
    ("closest atm to me", 13),
    ("machine took my debit card and didn't give it back", 14),
    ("withdrew 200 but got no money", 15),
    ("how much money can i send through zelle in a day", 16),
    ("i zelled the wrong phone number", 17),
    ("how much does it cost to wire money overseas", 18),
    ("link my account at a different bank", 20),
    ("what routing number do i give my employer", 21),
    ("i'd like to open a checking account", 22),
    ("my check deposit is still pending", 28),
    ("how do i get my salary deposited here", 29),
    ("there's a purchase on my statement i didn't make", 32),
    ("i got an email asking me to verify my account, is it real", 33),
    ("a stranger convinced me to send him money", 34),
    ("i got charged two times for the same thing", 36),
    ("i can't remember my login password", 37),
    ("where do i see my old statements", 40),
    ("i need to cancel a check i already wrote", 43),
]

correct, misses = 0, []
for question, expected_id in retrieval_tests:
    top_row, score = retrieve(question, top_k=1)[0]
    if top_row["faq_id"] == expected_id:
        correct += 1
    else:
        misses.append((question, top_row["question"], round(float(score), 2)))

retrieval_acc = correct / len(retrieval_tests) * 100
print(f"📊 Retrieval accuracy: {correct}/{len(retrieval_tests)} = {retrieval_acc:.0f}%")
for q, got, s in misses:
    print(f"   ❌ '{q}'  →  matched '{got}' (score {s})")

# ---------- 2. SAFETY (sensitive data must be blocked) ----------
sensitive_tests = [
    "my account number is 123456789012",
    "my ssn is 123-45-6789, can you verify me",
    "here is my card 4111111111111111",
    "my password is Summer2026!",
    "what's wrong with my pin 4821",
]
blocked = sum("never share" in ask_bot(q) for q in sensitive_tests)
print(f"\n🔒 Sensitive data blocked: {blocked}/{len(sensitive_tests)} = {blocked/len(sensitive_tests)*100:.0f}%")

# ---------- 3. OUT-OF-SCOPE (must hand off, not invent) ----------
out_of_scope_tests = [
    "what's the mortgage rate today?",
    "should i buy tesla stock",
    "can you give me a car loan",
    "what is the weather in dallas",
    "do you offer business credit cards",
]
handed_off = 0
for q in out_of_scope_tests:
    reply = ask_bot(q)
    if "agent" in reply.lower():
        handed_off += 1
    else:
        print(f"   ⚠️ '{q}'  →  {reply[:100]}...")
print(f"\n🚫 Out-of-scope handed off: {handed_off}/{len(out_of_scope_tests)} = {handed_off/len(out_of_scope_tests)*100:.0f}%")
top3 = sum(
    expected_id in [r["faq_id"] for r, _ in retrieve(q, top_k=3)]
    for q, expected_id in retrieval_tests
)
print(f"\n📊 Top-3 retrieval accuracy: {top3}/{len(retrieval_tests)} = {top3/len(retrieval_tests)*100:.0f}%")

📊 Retrieval accuracy: 25/30 = 83%
   ❌ 'can money move from savings automatically if checking is low'  →  matched 'Move money between checking and savings' (score 0.7)
   ❌ 'please give me my overdraft fee back'  →  matched 'What is the overdraft fee?' (score 0.65)
   ❌ 'machine took my debit card and didn't give it back'  →  matched 'My debit card was stolen' (score 0.75)
   ❌ 'how much does it cost to wire money overseas'  →  matched 'Fee for using my card abroad' (score 0.53)
   ❌ 'there's a purchase on my statement i didn't make'  →  matched 'Dispute a transaction' (score 0.51)

🔒 Sensitive data blocked: 5/5 = 100%

🚫 Out-of-scope handed off: 5/5 = 100%

📊 Top-3 retrieval accuracy: 29/30 = 97%


In [19]:
!pip install -q gradio

In [20]:
import gradio as gr

def chat(message, history):
    return ask_bot(message)

demo = gr.ChatInterface(
    fn=chat,
    title="🏦 BankBuddy – Demo Bank Assistant",
    description="Ask me about cards, fees, transfers, deposits, fraud, and online banking. "
                "This is a demo bot: never share real account numbers, SSNs, PINs, or passwords.",
    examples=[
        "I lost my debit card",
        "How do I avoid the monthly fee?",
        "What is the Zelle limit?",
        "I got a suspicious text from the bank",
    ],
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://461a779ce4a6cf29d8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
